# UniAV demo: cooking video → steps with captions

The model finds the steps of a cooking video (start, end) and gives each one a caption chosen from
8218 YouCook2 training sentences. Features: InternVideo2 (vision) + BEATs (audio).

The samples are YouCook2 **validation** videos (never seen in training) whose features are already
extracted, so each one runs in about a second. A GPU runtime is not needed for the samples.

**Before running:** add your HuggingFace token as a Colab secret named `HF_TOKEN`
(key icon on the left), with read access to `nguyenminh04/uniav-youcook2-data`.

In [ ]:
# 1. Code
!git clone -q --depth 1 -b test https://github.com/im-xiaoming/UniAV-fixed.git /content/UniAV
%cd /content/UniAV

In [ ]:
# 2. Model checkpoint (263 MB, private HF dataset -> needs the HF_TOKEN secret)
import os
from google.colab import userdata
from huggingface_hub import hf_hub_download
os.makedirs('ckpt/api', exist_ok=True)
hf_hub_download('nguyenminh04/uniav-youcook2-data', 'api/uniav_iv2.pth', repo_type='dataset',
                local_dir='ckpt', token=userdata.get('HF_TOKEN'))
!ls -la ckpt/api

In [ ]:
# 3. Load the model
import uniav_api as uv
pipe = uv.load()
print('model:', pipe.run, '| features:', pipe.spec.source, '| device:', pipe.device)
print('samples:', uv.samples())

## One video

`describe_sample` returns every event the model found: `start`, `end` (seconds), `caption`,
`score`, three alternative captions, and a 512-d vector used for search.
`show` / `plot` put each event next to the real step (YouCook2 annotation) it overlaps most:
**good** = IoU ≥ 0.5, **partial** = some overlap, **missed** = real step without a prediction,
**extra** = prediction without a real step.

In [ ]:
r = uv.describe_sample(uv.samples()[0])
uv.plot(r);

In [ ]:
uv.show(r)

In [ ]:
uv.compare_table(r)

In [ ]:
# raw result of one event
e = r['events'][0]
{k: v for k, v in e.items() if k != 'embedding'}

## Every sample

In [ ]:
import pandas as pd
from uniav_api.report import pair_events
rows = []
for vid in uv.samples():
    r = uv.describe_sample(vid)
    pairs, gt, preds = pair_events(r)
    rows.append({'video': vid, 'length (s)': round(r['duration']), 'real steps': len(gt),
                 'events found': len(preds), 'matched (IoU >= 0.5)': sum(p['verdict'] == 'good' for p in pairs)})
    uv.plot(r)
pd.DataFrame(rows)

## Text search over the videos described above

Without the 6 GB ONE-PEACE text encoder, a query is matched through the closest training captions
(word overlap), so phrase it like a recipe step.

In [ ]:
for q in ['cut the onion', 'add salt and pepper', 'fry the chicken', 'boil the noodles']:
    print('\n' + q)
    for h in uv.search(q, top_k=3):
        print('   %-12s %5.0f-%4.0f s  %-55s %.2f' % (h['video_id'], h['start'], h['end'], h['caption'][:55], h['score']))

## Optional: your own video (needs a GPU runtime)

Runs the InternVideo2 and BEATs encoders (about 3 GB of weights). The InternVideo2 model on
HuggingFace is gated: the token's account must have accepted its terms.

In [ ]:
!git clone -q --depth 1 https://github.com/OpenGVLab/InternVideo /content/UniAV/InternVideo
!pip install -q timm einops imageio-ffmpeg
for repo, name in [('OpenGVLab/InternVideo2-Stage2_1B-224p-f4', 'InternVideo2-stage2_1b-224p-f4.pt'),
                   ('OpenGVLab/InternVideo2-Stage2-6B-Audio', 'audio_6b.pth')]:
    hf_hub_download(repo, name, local_dir='ckpt/internvideo2', token=userdata.get('HF_TOKEN'))

In [ ]:
from google.colab import files
path = list(files.upload())[0]          # pick a cooking video (mp4)
r = uv.describe_video(path)
print('encode %.0f s, model %.2f s' % (r['timing']['encode_s'], r['timing']['model_s']))
for e in r['events']:
    print('%6.1f - %6.1f s  %s' % (e['start'], e['end'], e['caption']))